In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## Install and Imports

In [ ]:
%%capture
!pip install sentencepiece
!pip install transformers

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, T5Config, AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.optimization import AdamW
from tqdm import tqdm

## Model

In [ ]:
# Define the dataset class
class KeyTextDataset(Dataset):
    def __init__(self, keys, texts, tokenizer):
        self.keys = keys
        self.texts = texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        text = self.texts[idx]
        key_encoding = self.tokenizer(
            key,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
        )

        text_encoding = self.tokenizer(
            text,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
        )
        input_ids = key_encoding['input_ids'].squeeze()
        attention_mask = key_encoding['attention_mask'].squeeze()

        # print(text_encoding)

        labels = text_encoding['input_ids'].squeeze()
        labels[labels == 0] = -100
        labels_attention_mask = text_encoding['attention_mask'].squeeze()

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_attention_mask':labels_attention_mask,
            'text': text
        }

# Function to train the model
def train_model(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    # progress_bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        labels_attention_mask = batch['labels_attention_mask'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_attention_mask=labels_attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        # progress_bar.set_description(f"Train Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

# Function to validate the model
def validate_model(model, dataloader, device):
    model.eval()
    total_loss = 0

    # progress_bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        labels_attention_mask = batch['labels_attention_mask'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_attention_mask=labels_attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        # progress_bar.set_description(f"Train Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

# Function to save the trained model and tokenizer
def save_model(model, tokenizer, output_dir):
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Model and tokenizer saved to '{output_dir}'")

# Function to load the saved model and tokenizer
def load_model(output_dir):
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    print(f"Model and tokenizer loaded from '{output_dir}'")
    return model, tokenizer

In [ ]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model_dir = "csebuetnlp/mT5_multilingual_XLSum"
# model_dir = "csebuetnlp/banglat5"
# model_dir = "./Model/ModelV25"
model_dir = "./Model/bnT5ModelV25"
# Initialize the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
model.to(device)

## Dataset Load

In [ ]:
import pandas as pd
df =  pd.read_csv('./Data/Training Data/split2500K.csv')
df.columns = ["keywords", "text"]
df = df.reset_index(drop=True)
df

In [ ]:
print(df.isnull().sum())

keywords    0
text        0
dtype: int64


In [ ]:
df.dropna(inplace=True)

In [ ]:
print(df.isnull().sum())

keywords    0
text        0
dtype: int64


In [ ]:
# Load your dataset
keys = df['keywords'].tolist()  # List of keys
texts = df['text'].tolist()  # List of corresponding texts

In [ ]:
type(texts)

list

## Train

In [ ]:
# Split your dataset into train and validation sets
train_keys = keys[:int(len(keys)*0.8)]
train_texts = texts[:int(len(texts)*0.8)]
valid_keys = keys[int(len(keys)*0.8):]
valid_texts = texts[int(len(texts)*0.8):]

# Create the train and validation datasets
train_dataset = KeyTextDataset(train_keys, train_texts, tokenizer)
valid_dataset = KeyTextDataset(valid_keys, valid_texts, tokenizer)

# Create data loaders
batch_size = 2
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

# Set the training parameters
num_epochs = 1
learning_rate = 0.0001
warmup_steps = 500
total_steps = len(train_dataloader) * num_epochs

# Set the optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=learning_rate,
                                                total_steps=total_steps, div_factor=10,
                                                final_div_factor=100,
                                                pct_start=0.1,
                                                anneal_strategy='linear')

# Set the number of epochs for early stopping
patience = 3
best_valid_loss = float('inf')
epochs_no_improve = 0

# Start training
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    train_loss = train_model(model, tqdm(train_dataloader), optimizer, device)
    # print(f"Train Loss: {train_loss:.4f}")

    # Validate the model
    valid_loss = validate_model(model, tqdm(valid_dataloader), device)
    # print(f"Valid Loss: {valid_loss:.4f}")

    # Early stopping check
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print("Early stopping triggered!")
            break

    # Adjust the learning rate
    scheduler.step()
    # Print progress
    print('\n')
    print(f'Epoch: {epoch + 1} \tTraining Loss: {train_loss:.6f} \tValidation Loss: {valid_loss:.6f}')
    print('\n\n')

C:\Users\PC\anaconda3\envs\CSE_GPU_PYTORCH\lib\site-packages\transformers\optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/1


100%|██████████| 20000/20000 [30:54<00:00, 10.78it/s]



Epoch: 1 	Training Loss: 1.363261 	Validation Loss: 1.173621





In [ ]:
# Set the output directory for saving the model
# output_dir = "./Model/ModelV26"
output_dir = "./Model/bnT5ModelV26"

# Save the trained model and tokenizer
save_model(model, tokenizer, output_dir)

## Predictiion

In [ ]:
# loading_model_dir = "./Model/ModelV26"
loading_model_dir = "./Model/bnT5ModelV26"
loaded_model, loaded_tokenizer = load_model(loading_model_dir)
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)

In [ ]:
# Function to generate text given a key
def generate_text(key):
    input_ids = loaded_tokenizer.encode(key, return_tensors='pt',add_special_tokens=True).to(device)

    with torch.no_grad():
      outputs = loaded_model.generate(
          input_ids=input_ids,
          max_length =64,
          # max_new_tokens = 64,
          # num_beams =2,
          # num_beams = 1, # For Greedy
          # early_stopping =True,
          num_return_sequences = 1,
          temperature = 0.3,
          # top_k= 50,
          top_p= 0.95,
          do_sample=True,
          # do_sample=False, # For Greedy
          repetition_penalty= 2.5,
          length_penalty= 1.0)

    # print(outputs)

    preds = [loaded_tokenizer.decode(g,skip_special_tokens=True,clean_up_tokenization_spaces=True) for g in outputs]
    generated_text = preds[0]

    return generated_text

def predict(key):
  return generate_text(key)

In [ ]:
key = "নির্বাহের জীবিকা বৃদ্ধার সম্বল জমিটাই"
predict(key)

'বৃদ্ধার জীবিকা নির্বাহের একমাত্র সম্বল এই জমিটাই।'

In [ ]:
key = "ব্লগিং অনুপ্রেরণা অভ্যাস এর লেখালেখির আমার ছোটবেলার"
predict(key)

'এর পেছনে আমার ছোটবেলার লেখালেখির অভ্যাস, ব্লগিং ও লেখালেখিই অনুপ্রেরণা।'

In [ ]:
key = "দুই বিভিন্ন শ সাজা জনকে"
predict(key)

'বিভিন্ন মেয়াদে দুই শ জনকে সাজা দেওয়া হয়।'

In [ ]:
key = "প্রজন্ম চর্চায় সাহিত্য তরুণ"
predict(key)

'তরুণ প্রজন্ম সাহিত্য চর্চায় এগিয়ে আসছে।'

In [ ]:
key = "কেমন ডাটাসেট সময় ভাই বানাতে"
predict(key)

'ভাই, ডাটাসেট বানাতে কেমন সময় লাগে?'

In [ ]:
key = "আত্মা আমার দেখেই শুকিয়ে"
predict(key)

'দেখেই আমার আত্মা শুকিয়ে যায়।'

In [ ]:
key = "কেমন ডাটাসেট সময় বানাতে"
predict(key)

'ডাটাসেট বানাতে কেমন সময় লাগে?'

In [ ]:
key = "ঠিকমতো এই অভাবের তাই শিক্ষার্থীরা কারণে"
predict(key)

'তাই এই অভাবের কারণে শিক্ষার্থীরা ঠিকমতো পড়াশোনা করতে পারছে না।'